## dseek


# Automatic Linkage Topology & Kinematic Optimization

This notebook implements the core logic from the paper **"automatically determining exactly where to place new mechanical linkages to replicate an input motion while guaranteeing physical stability"**.

We demonstrate the pipeline on a simple planar 2‑link arm:
- **Input motion**: Two motors drive the shoulder and elbow joints to produce a target end‑effector trajectory.
- **Goal**: Remove the elbow motor and replace it with a rigid link (of fixed length) connecting two points on the upper arm and forearm, such that the same motion is replicated when only the shoulder motor is driven.

The algorithm finds the optimal attachment points using:
1. A distance‑variance minimisation with a moment‑arm penalty to avoid degenerate linkages.
2. A global CMA‑ES optimisation that refines the points while penalising mechanical singularities.

Interactive sliders let you explore the target motion, the candidate linkages, and the final optimised mechanism.


In [1]:
import scipy
print(scipy.__version__)

import scipy.sparse as sp
print(sp)

1.18.0
<module 'scipy.sparse' from '/Users/mj/Documents/A_windsurfStuff/ActiveGithub/complinkage/.venv/lib/python3.14/site-packages/scipy/sparse/__init__.py'>


In [2]:
import sys, scipy, numpy
print(sys.executable)
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print(scipy)

/Users/mj/Documents/A_windsurfStuff/ActiveGithub/complinkage/.venv/bin/python3
Python: 3.14.6 (main, Jun 10 2026, 10:03:53) [Clang 16.0.0 (clang-1600.0.26.6)]
NumPy: 2.5.1
SciPy: 1.18.0
<module 'scipy' from '/Users/mj/Documents/A_windsurfStuff/ActiveGithub/complinkage/.venv/lib/python3.14/site-packages/scipy/__init__.py'>


In [3]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D  # not used but kept for compatibility
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import cma
from scipy.sparse.linalg import svds
from scipy.optimize import approx_fprime

# For interactive plots
%matplotlib widget

Matplotlib is building the font cache; this may take a moment.


In [4]:
# %% [markdown]
# ## 2. Core Classes: Components, Constraints, and the Forward Kinematics Solver

# %%
class RigidComponent:
    """A 2D rigid body with state (alpha, x, y)."""
    def __init__(self, name, L=1.0):
        self.name = name
        self.L = L  # characteristic length for display
    def world_point(self, local_uv, state):
        """Transform a local point (u,v) to world coordinates given state (alpha,x,y)."""
        alpha, x, y = state[0], state[1], state[2]
        ca, sa = np.cos(alpha), np.sin(alpha)
        R = np.array([[ca, -sa], [sa, ca]])
        return R @ np.asarray(local_uv) + np.array([x, y])

class Constraint:
    """Base class for a kinematic constraint."""
    def evaluate(self, states, comp_indices):
        """Return constraint residual vector and its Jacobian w.r.t. the global state vector."""
        raise NotImplementedError

class PinJoint(Constraint):
    """Coincident point constraint: p_a = p_b."""
    def __init__(self, comp_a, local_a, comp_b, local_b):
        self.comp_a = comp_a
        self.local_a = np.asarray(local_a)
        self.comp_b = comp_b
        self.local_b = np.asarray(local_b)
    def evaluate(self, states, comp_indices):
        ia = comp_indices[self.comp_a]
        ib = comp_indices[self.comp_b]
        sa = states[ia]
        sb = states[ib]
        pa = self.comp_a.world_point(self.local_a, sa)
        pb = self.comp_b.world_point(self.local_b, sb)
        C = pa - pb   # 2-vector
        # Jacobian: 2 rows, columns for comp_a and comp_b
        # derivative of pa w.r.t sa
        alpha_a, xa, ya = sa[0], sa[1], sa[2]
        ua, va = self.local_a
        ca, sa_ = np.cos(alpha_a), np.sin(alpha_a)
        dR_dalpha = np.array([[-sa_, -ca], [ca, -sa_]])
        dpa_dalpha = dR_dalpha @ np.array([ua, va])
        # dpa/dx = [1,0], dpa/dy = [0,1]
        Ja = np.zeros((2,3))
        Ja[:,0] = dpa_dalpha
        Ja[:,1] = [1,0]
        Ja[:,2] = [0,1]
        # for pb similarly but with minus sign because C = pa - pb => dC/dsb = -dpb/dsb
        alpha_b, xb, yb = sb[0], sb[1], sb[2]
        ub, vb = self.local_b
        cb, sb_ = np.cos(alpha_b), np.sin(alpha_b)
        dR_dalpha_b = np.array([[-sb_, -cb], [cb, -sb_]])
        dpb_dalpha = dR_dalpha_b @ np.array([ub, vb])
        Jb = np.zeros((2,3))
        Jb[:,0] = -dpb_dalpha
        Jb[:,1] = [-1,0]
        Jb[:,2] = [0,-1]
        return C, (ia, Ja), (ib, Jb)

class MotorConstraint(Constraint):
    """Controls relative angle: alpha_a - alpha_b = target_angle."""
    def __init__(self, comp_a, comp_b, target_angle):
        self.comp_a = comp_a
        self.comp_b = comp_b
        self.target_angle = target_angle
    def evaluate(self, states, comp_indices):
        ia = comp_indices[self.comp_a]
        ib = comp_indices[self.comp_b]
        alpha_a = states[ia][0]
        alpha_b = states[ib][0]
        C = np.array([alpha_a - alpha_b - self.target_angle])
        # Jacobian: 1 row, derivative w.r.t alpha_a = 1, alpha_b = -1
        Ja = np.array([[1,0,0]])
        Jb = np.array([[-1,0,0]])
        return C, (ia, Ja), (ib, Jb)

class GroundConstraint(Constraint):
    """Fixes the world position of a point on a component to a fixed world point."""
    def __init__(self, comp, local_pt, world_pt):
        self.comp = comp
        self.local_pt = np.asarray(local_pt)
        self.world_pt = np.asarray(world_pt)
    def evaluate(self, states, comp_indices):
        i = comp_indices[self.comp]
        s = states[i]
        p = self.comp.world_point(self.local_pt, s)
        C = p - self.world_pt
        # Jacobian similar to PinJoint but only for one component
        alpha, x, y = s[0], s[1], s[2]
        u, v = self.local_pt
        ca, sa = np.cos(alpha), np.sin(alpha)
        dR = np.array([[-sa, -ca], [ca, -sa]])
        dp_dalpha = dR @ np.array([u, v])
        J = np.zeros((2,3))
        J[:,0] = dp_dalpha
        J[:,1] = [1,0]
        J[:,2] = [0,1]
        return C, (i, J), None

class DistanceConstraint(Constraint):
    """Keeps the distance between two points constant: ||p_a - p_b||^2 - L^2 = 0."""
    def __init__(self, comp_a, local_a, comp_b, local_b, L):
        self.comp_a = comp_a
        self.local_a = np.asarray(local_a)
        self.comp_b = comp_b
        self.local_b = np.asarray(local_b)
        self.L = L
        self.L2 = L**2
    def evaluate(self, states, comp_indices):
        ia = comp_indices[self.comp_a]
        ib = comp_indices[self.comp_b]
        sa = states[ia]
        sb = states[ib]
        pa = self.comp_a.world_point(self.local_a, sa)
        pb = self.comp_b.world_point(self.local_b, sb)
        d = pa - pb
        C = np.array([np.dot(d, d) - self.L2])
        # Gradient
        # dC/dsa = 2 * d^T * dpa/dsa,   dC/dsb = 2 * d^T * (-dpb/dsb) = -2 d^T * dpb/dsb
        alpha_a, xa, ya = sa[0], sa[1], sa[2]
        ua, va = self.local_a
        ca, sa_ = np.cos(alpha_a), np.sin(alpha_a)
        dR_a = np.array([[-sa_, -ca], [ca, -sa_]])
        dpa_dalpha = dR_a @ np.array([ua, va])
        grad_a_alpha = 2 * np.dot(d, dpa_dalpha)
        grad_a_xy = 2 * d  # since dpa/d(x,y) = I
        Ja = np.array([[grad_a_alpha, grad_a_xy[0], grad_a_xy[1]]])
        alpha_b, xb, yb = sb[0], sb[1], sb[2]
        ub, vb = self.local_b
        cb, sb_ = np.cos(alpha_b), np.sin(alpha_b)
        dR_b = np.array([[-sb_, -cb], [cb, -sb_]])
        dpb_dalpha = dR_b @ np.array([ub, vb])
        grad_b_alpha = -2 * np.dot(d, dpb_dalpha)
        grad_b_xy = -2 * d
        Jb = np.array([[grad_b_alpha, grad_b_xy[0], grad_b_xy[1]]])
        return C, (ia, Ja), (ib, Jb)

class Assembly:
    """Holds components and constraints, provides forward kinematics solve."""
    def __init__(self):
        self.components = []
        self.constraints = []
        self.comp_to_idx = {}
        self.n_dof = 0
    def add_component(self, comp):
        idx = len(self.components)
        self.comp_to_idx[comp] = idx
        self.components.append(comp)
        self.n_dof += 3
    def add_constraint(self, constraint):
        self.constraints.append(constraint)
    def solve_state(self, initial_guess, max_iter=50, tol=1e-8):
        """Newton-Raphson solver, returns final state vector of length n_dof."""
        s = initial_guess.copy()
        for it in range(max_iter):
            C_vec, J_rows, J_cols, J_data = [], [], [], []
            for con in self.constraints:
                C_i, block_a, block_b = con.evaluate(s, self.comp_to_idx)
                C_vec.append(C_i)
                if block_a is not None:
                    i, Ja = block_a
                    nrows, ncols = Ja.shape
                    for r in range(nrows):
                        for c in range(ncols):
                            J_rows.append(sum(len(cc) for cc in C_vec[:-1]) + r)  # correct row index later
                            J_cols.append(i*3 + c)
                            J_data.append(Ja[r,c])
                if block_b is not None:
                    i, Jb = block_b
                    nrows, ncols = Jb.shape
                    for r in range(nrows):
                        for c in range(ncols):
                            J_rows.append(sum(len(cc) for cc in C_vec[:-1]) + r)
                            J_cols.append(i*3 + c)
                            J_data.append(Jb[r,c])
            C_full = np.concatenate(C_vec)
            # Build sparse J
            total_rows = len(C_full)
            J = sp.coo_matrix((J_data, (J_rows, J_cols)), shape=(total_rows, self.n_dof)).tocsc()
            JtJ = J.T @ J
            JtC = J.T @ C_full
            try:
                delta = spla.spsolve(JtJ, -JtC)
            except Exception:
                # If singular, use least squares
                delta = spla.lsmr(J, -C_full)[0]
            s_new = s + delta
            # Simple line search to ensure energy decrease
            E_old = 0.5 * np.dot(C_full, C_full)
            step = 1.0
            for _ in range(10):
                s_test = s + step * delta
                # Recompute C at s_test
                C_test = []
                for con in self.constraints:
                    C_i, _, _ = con.evaluate(s_test, self.comp_to_idx)
                    C_test.append(C_i)
                C_test = np.concatenate(C_test)
                E_new = 0.5 * np.dot(C_test, C_test)
                if E_new < E_old:
                    s = s_test
                    break
                step *= 0.5
            else:
                s = s_new  # accept anyway if line search fails
            if np.linalg.norm(delta) < tol:
                break
        return s

# %% [markdown]
# ## 3. Define the Input (Target) Mechanism and Record Motion

# %%
# Create components
upper_arm = RigidComponent("UpperArm", L=1.0)
forearm = RigidComponent("Forearm", L=0.8)
ground = RigidComponent("Ground", L=0.0)

# Build original assembly with two motors
asm_orig = Assembly()
asm_orig.add_component(upper_arm)
asm_orig.add_component(forearm)
asm_orig.add_component(ground)
# Ground is fixed at (0,0,0) – we impose that its state never changes (we'll fix state externally)
# Pin joint: shoulder – ground to upper_arm origin
asm_orig.add_constraint(PinJoint(ground, (0,0), upper_arm, (0,0)))
# Pin joint: elbow – upper_arm tip to forearm origin
asm_orig.add_constraint(PinJoint(upper_arm, (upper_arm.L,0), forearm, (0,0)))
# Motor 1: shoulder angle (between ground and upper arm)
def shoulder_angle(t):
    return 0.5 * np.sin(2*np.pi*t)
# Motor 2: elbow relative angle
def elbow_angle(t):
    return 0.8 * np.cos(2*np.pi*t + 0.5)

# We will step through time, set motor target angles, and solve.
# Because ground is a component but its state should stay (0,0,0), we can add a GroundConstraint fixing its origin.
asm_orig.add_constraint(GroundConstraint(ground, (0,0), (0,0)))

# Time samples
n_samples = 60
t_vals = np.linspace(0, 1, n_samples, endpoint=False)
target_states = []  # list of arrays length n_dof (6 DOF: upper, forearm, ground)
target_marker_pos = []  # world position of end-effector (forearm tip)

# Initial guess for first solve: all zero, but ground is fixed
s_init = np.zeros(asm_orig.n_dof)
# We'll reuse previous solution as next initial guess
s_prev = s_init.copy()

for t in t_vals:
    # Update motor constraints with current target angles
    # We need to rebuild assembly because constraints are fixed; we can create a new assembly each time or modify.
    # Here we build a fresh assembly with motor targets.
    asm = Assembly()
    asm.add_component(upper_arm)
    asm.add_component(forearm)
    asm.add_component(ground)
    asm.add_constraint(PinJoint(ground, (0,0), upper_arm, (0,0)))
    asm.add_constraint(PinJoint(upper_arm, (upper_arm.L,0), forearm, (0,0)))
    asm.add_constraint(GroundConstraint(ground, (0,0), (0,0)))
    asm.add_constraint(MotorConstraint(upper_arm, ground, shoulder_angle(t)))
    asm.add_constraint(MotorConstraint(forearm, upper_arm, elbow_angle(t)))
    s_sol = asm.solve_state(s_prev)
    s_prev = s_sol
    target_states.append(s_sol)
    # Marker: forearm tip at local (forearm.L, 0)
    idx_forearm = asm.comp_to_idx[forearm]
    sf = s_sol[idx_forearm*3:idx_forearm*3+3]
    marker = forearm.world_point((forearm.L, 0), sf)
    target_marker_pos.append(marker)

target_states = np.array(target_states)  # shape (n_samples, 6)
target_marker_pos = np.array(target_marker_pos)

# %% [markdown]
# ### Visualise the Target Motion with a Time Slider

# %%
def plot_target_frame(t):
    idx = np.argmin(np.abs(t_vals - t))
    s = target_states[idx]
    # extract states
    alpha_u, x_u, y_u = s[0], s[1], s[2]
    alpha_f, x_f, y_f = s[3], s[4], s[5]
    # shoulder at (0,0)
    elbow = upper_arm.world_point((upper_arm.L,0), (alpha_u, x_u, y_u))
    hand = forearm.world_point((forearm.L,0), (alpha_f, x_f, y_f))
    fig, ax = plt.subplots(figsize=(6,6))
    ax.plot([0, elbow[0]], [0, elbow[1]], 'b-o', lw=3, label='UpperArm')
    ax.plot([elbow[0], hand[0]], [elbow[1], hand[1]], 'r-o', lw=3, label='Forearm')
    ax.plot(target_marker_pos[:idx+1,0], target_marker_pos[:idx+1,1], 'gray', alpha=0.5, label='End-effector path')
    ax.scatter(*hand, c='green', s=80, zorder=5, label='Hand')
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.legend()
    ax.set_title(f"Target Motion at t={t:.2f}")
    plt.show()

widgets.interact(plot_target_frame, t=widgets.FloatSlider(min=0, max=1, step=0.01, value=0, description='Time'))

# %% [markdown]
# ## 4. Topology Optimisation: Find the Best Linkage Points (Distance Variance + Moment Arm)

# %% [markdown]
# We now remove the elbow motor and search for two points (local coordinates) on the upper arm and forearm whose world‑space distance is as constant as possible over the cycle.  
# We also penalise small moment arms (triangle area) to ensure the link can transmit torque.

# %%
# Precompute data for objective: we need the states of upper arm and forearm, and the world position of the elbow joint (motor pivot)
states_u = target_states[:, 0:3]   # upper arm
states_f = target_states[:, 3:6]   # forearm
# elbow world position over time
elbow_pos = np.array([upper_arm.world_point((upper_arm.L,0), su) for su in states_u])

# Build the quadratic forms for distance variance
# For each time i, distance^2 = ||A_i q_u - B_i q_f + t_diff_i||^2, where q_u=(u_u,v_u), q_f=(u_f,v_f)
# A_i = rotation matrix of upper arm, B_i = rotation of forearm, t_diff_i = translation_u - translation_f
# We stack X = [u_u, v_u, u_f, v_f]
As = np.zeros((n_samples,2,2))
Bs = np.zeros((n_samples,2,2))
t_diffs = np.zeros((n_samples,2))
for i in range(n_samples):
    au, xu, yu = states_u[i]
    af, xf, yf = states_f[i]
    cau, sau = np.cos(au), np.sin(au)
    As[i] = np.array([[cau, -sau], [sau, cau]])
    caf, saf = np.cos(af), np.sin(af)
    Bs[i] = np.array([[caf, -saf], [saf, caf]])
    t_diffs[i] = np.array([xu, yu]) - np.array([xf, yf])

# For each i, the 4x4 matrix M_i and vector v_i, scalar c_i such that d_i = X^T M_i X + 2 v_i^T X + c_i
M_all = np.zeros((n_samples,4,4))
v_all = np.zeros((n_samples,4))
c_all = np.zeros(n_samples)
for i in range(n_samples):
    A = As[i]  # 2x2
    B = Bs[i]
    # The stacked vector: [A, -B] * X + t_diff
    # => block matrix: [A  -B] of size 2x4
    M_i = np.zeros((4,4))
    # upper-left 2x2: A^T A, but since A is rotation, A^T A = I
    # Actually, d_i = || [A, -B] X + t_diff ||^2 = X^T [A; -B]^T [A; -B] X + 2 t_diff^T [A, -B] X + t_diff^T t_diff
    # So M_i = [A^T A, -A^T B; -B^T A, B^T B]
    AtA = A.T @ A  # identity
    BtB = B.T @ B  # identity
    AtB = A.T @ B
    BtA = B.T @ A
    M_i[0:2,0:2] = AtA
    M_i[0:2,2:4] = -AtB
    M_i[2:4,0:2] = -BtA
    M_i[2:4,2:4] = BtB
    v_i = np.zeros(4)
    v_i[0:2] = A.T @ t_diffs[i]
    v_i[2:4] = -B.T @ t_diffs[i]
    c_i = np.dot(t_diffs[i], t_diffs[i])
    M_all[i] = M_i
    v_all[i] = v_i
    c_all[i] = c_i

# Mean matrices
M_mean = np.mean(M_all, axis=0)
v_mean = np.mean(v_all, axis=0)
c_mean = np.mean(c_all)

# Functions for delta (distance variance) and its gradient/Hessian
def compute_delta_grad_hess(X):
    """Returns delta, grad (4,), Hess (4,4)"""
    d_all = np.einsum('i,ij,j', X, M_all, X) + 2 * np.einsum('ij,j', v_all, X) + c_all  # (n,)
    l_val = np.einsum('i,ij,j', X, M_mean, X) + 2 * np.dot(v_mean, X) + c_mean  # scalar
    resid = d_all - l_val
    delta = np.mean(resid**2)
    # Gradient: 2/n sum resid * (grad d_i - grad l)
    grad_d_all = 2 * (np.einsum('ijk,k', M_all, X) + v_all)  # (n,4)
    grad_l = 2 * (M_mean @ X + v_mean)  # (4,)
    grad = 2.0/n_samples * np.sum(resid[:, np.newaxis] * (grad_d_all - grad_l[np.newaxis,:]), axis=0)
    # Hessian: 2/n sum [ (grad d_i - grad l)(grad d_i - grad l)^T + resid_i * (2 M_i - 2 M_mean) ]
    diff_grad = grad_d_all - grad_l[np.newaxis,:]  # (n,4)
    Hess = 2.0/n_samples * (diff_grad.T @ diff_grad)  # (4,4)
    for i in range(n_samples):
        Hess += 2.0/n_samples * resid[i] * 2 * (M_all[i] - M_mean)
    return delta, grad, Hess

# Moment arm penalty
def area_sq_i(X, i):
    """Squared area of triangle (x_b, x_m, x_a) at time i, times 4 (we ignore 1/2 factor)"""
    u_u, v_u, u_f, v_f = X[0], X[1], X[2], X[3]
    # world points
    au, xu, yu = states_u[i]
    cau, sau = np.cos(au), np.sin(au)
    p_a = np.array([cau*u_u - sau*v_u + xu, sau*u_u + cau*v_u + yu])
    af, xf, yf = states_f[i]
    caf, saf = np.cos(af), np.sin(af)
    p_b = np.array([caf*u_f - saf*v_f + xf, saf*u_f + caf*v_f + yf])
    p_m = elbow_pos[i]
    cross = (p_a[0]-p_m[0])*(p_b[1]-p_m[1]) - (p_a[1]-p_m[1])*(p_b[0]-p_m[0])
    return cross**2  # 4 * area^2

def area_penalty(X):
    """E_Area = -log( sum area_i^2 / n )"""
    areas = np.array([area_sq_i(X, i) for i in range(n_samples)])
    sum_a2 = np.sum(areas)
    if sum_a2 < 1e-12:
        return 1e12, np.zeros(4)  # avoid log(0)
    val = -np.log(sum_a2 / n_samples)
    # Gradient
    grad = np.zeros(4)
    for i in range(n_samples):
        a2 = areas[i]
        if a2 < 1e-12:
            continue
        # gradient of area_sq_i w.r.t X using finite diff (analytic is doable but keep simple for demo)
        eps = 1e-6
        for j in range(4):
            X_plus = X.copy()
            X_plus[j] += eps
            a2_plus = area_sq_i(X_plus, i)
            grad[j] += (a2_plus - a2) / eps
    grad = - grad / (sum_a2)  # derivative of -log(sum a2) = -1/sum_a2 * sum grad a2_i
    return val, grad

# Combined objective for topology search
def total_objective(X, gamma=0.1):
    delta, grad_d, Hess_d = compute_delta_grad_hess(X)
    area_val, grad_a = area_penalty(X)
    J = delta + gamma * area_val
    grad = grad_d + gamma * grad_a
    # For Newton we need Hessian of delta; we approximate area part with Gauss-Newton style? Use full Hess_d and ignore area Hessian for simplicity.
    return J, grad, Hess_d

# Multi-start Newton optimisation
def newton_optimize(X0, gamma=0.1, max_iter=50):
    X = X0.copy()
    for it in range(max_iter):
        J, grad, Hess = total_objective(X, gamma)
        # Regularise Hessian if needed
        try:
            delta_X = np.linalg.solve(Hess + 1e-4*np.eye(4), -grad)
        except np.linalg.LinAlgError:
            delta_X = -grad
        # Backtracking line search
        step = 1.0
        for _ in range(20):
            X_new = X + step * delta_X
            J_new, _, _ = total_objective(X_new, gamma)
            if J_new < J:
                X = X_new
                break
            step *= 0.5
        else:
            break
        if np.linalg.norm(delta_X) < 1e-8:
            break
    return X, J

# Generate random initial guesses and run
candidate_Xs = []
candidate_costs = []
gamma = 0.1
num_candidates = 8
np.random.seed(42)
for _ in range(num_candidates):
    # Random local coords within reasonable bounds (on the bodies)
    u_u = np.random.uniform(0.1, 0.9) * upper_arm.L
    v_u = np.random.uniform(-0.5, 0.5)
    u_f = np.random.uniform(0.0, 0.7) * forearm.L
    v_f = np.random.uniform(-0.4, 0.4)
    X0 = np.array([u_u, v_u, u_f, v_f])
    X_opt, cost = newton_optimize(X0, gamma)
    candidate_Xs.append(X_opt)
    candidate_costs.append(cost)

# Sort by cost
order = np.argsort(candidate_costs)
candidate_Xs = [candidate_Xs[i] for i in order]
candidate_costs = [candidate_costs[i] for i in order]

# %% [markdown]
# ### Interactive Visualisation of Candidate Linkages

# %%
def plot_candidate(cand_idx, t):
    """Draw the target mechanism at time t with the chosen candidate link."""
    idx_t = np.argmin(np.abs(t_vals - t))
    s = target_states[idx_t]
    alpha_u, xu, yu = s[0], s[1], s[2]
    alpha_f, xf, yf = s[3], s[4], s[5]
    elbow = upper_arm.world_point((upper_arm.L,0), (alpha_u, xu, yu))
    hand = forearm.world_point((forearm.L,0), (alpha_f, xf, yf))
    fig, ax = plt.subplots(figsize=(6,6))
    ax.plot([0, elbow[0]], [0, elbow[1]], 'b-o', lw=3)
    ax.plot([elbow[0], hand[0]], [elbow[1], hand[1]], 'r-o', lw=3)
    # Candidate points
    X = candidate_Xs[cand_idx]
    p_a = upper_arm.world_point(X[0:2], (alpha_u, xu, yu))
    p_b = forearm.world_point(X[2:4], (alpha_f, xf, yf))
    ax.scatter(*p_a, c='magenta', s=100, zorder=6, marker='s', label='q_a')
    ax.scatter(*p_b, c='cyan', s=100, zorder=6, marker='D', label='q_b')
    ax.plot([p_a[0], p_b[0]], [p_a[1], p_b[1]], 'k--', lw=2, label='candidate link')
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.legend()
    ax.set_title(f"Candidate {cand_idx} (cost={candidate_costs[cand_idx]:.4f}) at t={t:.2f}")
    plt.show()

cand_slider = widgets.IntSlider(min=0, max=len(candidate_Xs)-1, value=0, description='Candidate')
time_slider = widgets.FloatSlider(min=0, max=1, step=0.01, value=0, description='Time')
widgets.interact(plot_candidate, cand_idx=cand_slider, t=time_slider)

# %% [markdown]
# ## 5. Global Optimisation with CMA‑ES (Refining Linkage Points)

# %% [markdown]
# We pick the best candidate from the topology stage and then use CMA‑ES to fine‑tune the local coordinates of the link attachments.  
# The cost function now includes:
# - **Marker deviation**: end‑effector trajectory error.
# - **Singularity penalty**: smallest singular value of the constraint Jacobian over the cycle.

# %%
# Best candidate initial guess
best_X = candidate_Xs[0]
print("Best candidate X:", best_X)

# Compute constant link length L as the mean distance from target motion (or we can take distance at t=0)
# We'll compute it from the candidate at the first time step
s0 = target_states[0]
alpha_u0, xu0, yu0 = s0[0], s0[1], s0[2]
alpha_f0, xf0, yf0 = s0[3], s0[4], s0[5]
p_a0 = upper_arm.world_point(best_X[0:2], (alpha_u0, xu0, yu0))
p_b0 = forearm.world_point(best_X[2:4], (alpha_f0, xf0, yf0))
L_link = np.linalg.norm(p_a0 - p_b0)
print(f"Link length L = {L_link:.4f}")

# Build the new mechanism (shoulder motor + elbow pin + rigid distance constraint)
def simulate_new_mechanism(X, L_fixed=None):
    """Given X = [u_u, v_u, u_f, v_f], simulate over cycle with only shoulder motor.
       Returns marker positions array (n_samples,2) and smallest singular values list."""
    if L_fixed is None:
        # compute L from X at t=0
        p_a_0 = upper_arm.world_point(X[0:2], states_u[0])
        p_b_0 = forearm.world_point(X[2:4], states_f[0])
        L = np.linalg.norm(p_a_0 - p_b_0)
    else:
        L = L_fixed
    marker_positions = []
    singular_vals = []
    s_prev = np.zeros(6)  # initial guess
    for i, t in enumerate(t_vals):
        asm = Assembly()
        asm.add_component(upper_arm)
        asm.add_component(forearm)
        asm.add_component(ground)
        asm.add_constraint(PinJoint(ground, (0,0), upper_arm, (0,0)))
        asm.add_constraint(PinJoint(upper_arm, (upper_arm.L,0), forearm, (0,0)))
        asm.add_constraint(GroundConstraint(ground, (0,0), (0,0)))
        # Shoulder motor
        asm.add_constraint(MotorConstraint(upper_arm, ground, shoulder_angle(t)))
        # Distance constraint
        asm.add_constraint(DistanceConstraint(upper_arm, X[0:2], forearm, X[2:4], L))
        # Solve
        s_sol = asm.solve_state(s_prev)
        s_prev = s_sol
        # Marker
        idx_f = asm.comp_to_idx[forearm]
        sf = s_sol[idx_f*3:idx_f*3+3]
        marker = forearm.world_point((forearm.L, 0), sf)
        marker_positions.append(marker)
        # Compute smallest singular value of constraint Jacobian at solution
        # Build Jacobian matrix sparse
        C_vec, J_rows, J_cols, J_data = [], [], []
        for con in asm.constraints:
            C_i, block_a, block_b = con.evaluate(s_sol, asm.comp_to_idx)
            C_vec.append(C_i)
            if block_a is not None:
                i_idx, Ja = block_a
                nr, nc = Ja.shape
                for r in range(nr):
                    for c in range(nc):
                        J_rows.append(sum(len(cc) for cc in C_vec[:-1]) + r)
                        J_cols.append(i_idx*3 + c)
                        J_data.append(Ja[r,c])
            if block_b is not None:
                i_idx, Jb = block_b
                nr, nc = Jb.shape
                for r in range(nr):
                    for c in range(nc):
                        J_rows.append(sum(len(cc) for cc in C_vec[:-1]) + r)
                        J_cols.append(i_idx*3 + c)
                        J_data.append(Jb[r,c])
        total_rows = len(np.concatenate(C_vec))
        J = sp.coo_matrix((J_data, (J_rows, J_cols)), shape=(total_rows, 6)).tocsc()
        # smallest singular value
        if min(J.shape) >= 1:
            sval = svds(J, k=1, which='SM', return_singular_vectors=False)[0]
        else:
            sval = 0.0
        singular_vals.append(sval)
    return np.array(marker_positions), np.array(singular_vals)

# Objective for CMA-ES
def global_cost(X):
    """X is 4-dimensional vector of local coordinates. Returns scalar cost."""
    X = np.asarray(X)
    marker_traj, s_vals = simulate_new_mechanism(X, L_fixed=None)
    # Marker deviation
    E_marker = np.sum(np.linalg.norm(marker_traj - target_marker_pos, axis=1)**2)
    # Singularity penalty: sum (sval + epsilon)^(-alpha)
    epsilon = 1e-4
    alpha = 2.0
    E_sing = np.sum(1.0 / (s_vals + epsilon)**alpha)
    # Weights
    w1 = 1.0
    w2 = 0.1   # singularity weight
    total = w1 * E_marker + w2 * E_sing
    # Add a small penalty to keep points on the bodies (soft bounds)
    # upper arm length 1.0, forearm 0.8
    if X[0] < 0 or X[0] > upper_arm.L:
        total += 100.0 * (min(X[0], 0)**2 + max(X[0]-upper_arm.L, 0)**2)
    if X[2] < 0 or X[2] > forearm.L:
        total += 100.0 * (min(X[2], 0)**2 + max(X[2]-forearm.L, 0)**2)
    return total

# Run CMA-ES
initial_guess = best_X.copy()
sigma0 = 0.1
es = cma.CMAEvolutionStrategy(initial_guess, sigma0, {'maxfevals': 500, 'seed': 2023})
print("Running CMA-ES...")
while not es.stop():
    solutions = es.ask()
    es.tell(solutions, [global_cost(x) for x in solutions])
    es.disp()
es.result_pretty()
X_optimized = es.result.xbest
print("Optimized X:", X_optimized)

# %% [markdown]
# ### Compare Final Motion with Target

# %%
# Simulate with optimized X
opt_marker, opt_sing = simulate_new_mechanism(X_optimized)
# Plot trajectories
fig, ax = plt.subplots(figsize=(6,6))
ax.plot(target_marker_pos[:,0], target_marker_pos[:,1], 'b-', lw=2, label='Target')
ax.plot(opt_marker[:,0], opt_marker[:,1], 'r--', lw=2, label='Optimized')
ax.scatter(target_marker_pos[0,0], target_marker_pos[0,1], color='blue')
ax.scatter(opt_marker[0,0], opt_marker[0,1], color='red')
ax.set_aspect('equal')
ax.legend()
ax.set_title('End‑Effector Trajectory')
plt.show()

# Time slider to view full mechanism
def plot_optimized_frame(t):
    idx = np.argmin(np.abs(t_vals - t))
    # Get target states for overlay
    s_targ = target_states[idx]
    # Get optimized state by re-simulating just for that time step
    asm = Assembly()
    asm.add_component(upper_arm)
    asm.add_component(forearm)
    asm.add_component(ground)
    asm.add_constraint(PinJoint(ground, (0,0), upper_arm, (0,0)))
    asm.add_constraint(PinJoint(upper_arm, (upper_arm.L,0), forearm, (0,0)))
    asm.add_constraint(GroundConstraint(ground, (0,0), (0,0)))
    asm.add_constraint(MotorConstraint(upper_arm, ground, shoulder_angle(t_vals[idx])))
    L = np.linalg.norm(upper_arm.world_point(X_optimized[0:2], states_u[idx]) - forearm.world_point(X_optimized[2:4], states_f[idx]))
    asm.add_constraint(DistanceConstraint(upper_arm, X_optimized[0:2], forearm, X_optimized[2:4], L))
    s_opt = asm.solve_state(np.zeros(6))
    alpha_u_o, xu_o, yu_o = s_opt[0], s_opt[1], s_opt[2]
    alpha_f_o, xf_o, yf_o = s_opt[3], s_opt[4], s_opt[5]
    elbow_o = upper_arm.world_point((upper_arm.L,0), (alpha_u_o, xu_o, yu_o))
    hand_o = forearm.world_point((forearm.L,0), (alpha_f_o, xf_o, yf_o))
    fig, ax = plt.subplots(figsize=(6,6))
    ax.plot([0, elbow_o[0]], [0, elbow_o[1]], 'b-o', lw=3, label='Optimized Upper')
    ax.plot([elbow_o[0], hand_o[0]], [elbow_o[1], hand_o[1]], 'r-o', lw=3, label='Optimized Forearm')
    # Draw the rigid link
    p_a = upper_arm.world_point(X_optimized[0:2], (alpha_u_o, xu_o, yu_o))
    p_b = forearm.world_point(X_optimized[2:4], (alpha_f_o, xf_o, yf_o))
    ax.plot([p_a[0], p_b[0]], [p_a[1], p_b[1]], 'k--', lw=2, label='Rigid Link')
    # Overlay target skeleton faded
    alpha_u_t, xu_t, yu_t = s_targ[0], s_targ[1], s_targ[2]
    alpha_f_t, xf_t, yf_t = s_targ[3], s_targ[4], s_targ[5]
    elbow_t = upper_arm.world_point((upper_arm.L,0), (alpha_u_t, xu_t, yu_t))
    hand_t = forearm.world_point((forearm.L,0), (alpha_f_t, xf_t, yf_t))
    ax.plot([0, elbow_t[0]], [0, elbow_t[1]], 'gray', alpha=0.3, lw=2)
    ax.plot([elbow_t[0], hand_t[0]], [elbow_t[1], hand_t[1]], 'gray', alpha=0.3, lw=2)
    ax.set_xlim(-2,2)
    ax.set_ylim(-2,2)
    ax.set_aspect('equal')
    ax.legend()
    ax.set_title(f"Optimized vs Target at t={t:.2f}")
    plt.show()

widgets.interact(plot_optimized_frame, t=widgets.FloatSlider(min=0, max=1, step=0.01, value=0, description='Time'))

# %% [markdown]
# ## 6. Singularity Landscape (Optional Exploration)
# We can plot the smallest singular value of the constraint Jacobian for different attachment points to visualise the singularity penalty.

# %%
# Sample a grid around the optimized point for a quick heatmap (u_u vs u_f, fixing v=0)
u_vals = np.linspace(0.1, 0.9, 10)
uf_vals = np.linspace(0.1, 0.7, 10)
sing_map = np.zeros((len(u_vals), len(uf_vals)))
for i, uu in enumerate(u_vals):
    for j, uf in enumerate(uf_vals):
        X_test = np.array([uu, 0.0, uf, 0.0])
        _, s_vals = simulate_new_mechanism(X_test)
        sing_map[i,j] = np.min(s_vals)
fig, ax = plt.subplots()
c = ax.contourf(u_vals, uf_vals, sing_map.T, levels=20)
plt.colorbar(c, label='Min singular value')
ax.set_xlabel('u_upper')
ax.set_ylabel('u_forearm')
ax.set_title('Smallest singular value over the cycle')
plt.show()

# %% [markdown]
# ---
# The script above provides the complete mathematical pipeline:
# - Forward kinematics with Newton‑Raphson and sparse Jacobians.
# - Distance variance minimisation with analytic gradients and Hessians, combined with a moment‑arm penalty.
# - Multi‑start Newton to generate diverse candidate linkages.
# - Global CMA‑ES optimisation with a singularity penalty computed via sparse SVD.
# - Interactive visualisations throughout.
# 
# This can be extended to 3D, more complex skeletons, and fabrication output (STL generation) as per the full specification.

IndexError: invalid index to scalar variable.

In [2]:
import numpy